In [27]:
import pandas as pd

In [28]:
twitter = pd.read_csv('df_twitter.csv')

In [29]:
twitter.drop(columns=['Content_cleaned', 'Content_cleaned_2', 'In Reply To Normalized',
                      'Author_Normalized', ], inplace=True)

In [30]:
twitter.columns

Index(['Author', 'Content', 'Date', 'Location', 'Number of Likes',
       'Number of Retweets', 'In Reply To', 'Author Name',
       'Author Description', 'Author Statuses Count',
       'Author Favourites Count', 'Author Friends Count',
       'Author Followers Count', 'Author Listed Count', 'Author Verified',
       'Mentions', 'Hashtags', 'Seed_Set_1', 'Seed_Set_2', 'Seed_Set_2_Prob',
       'Seed_Set_2_Model', 'Seed_Set_3', 'Entidad', 'Prob entidad',
       'Polaridad', 'Fecha', 'Sentimiento'],
      dtype='object')

In [31]:
"""
Segun la expo de Karen:

Raw text -> Tokenization -> Normalization -> Noise removal -> Stop word removal 
-> Lemmatization/Stemming -> Processed text
"""

'\nSegun la expo de Karen:\n\nRaw text -> Tokenization -> Normalization -> Noise removal -> Stop word removal \n-> Lemmatization/Stemming -> Processed text\n'

In [32]:
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm
import gc

# ============================================
# 1. VERIFICAR GPU Y CONFIGURAR
# ============================================

print("🔍 Verificando hardware disponible...")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU detectada: {torch.cuda.get_device_name(0)}")
    
    # Configurar para mejor rendimiento
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    
else:
    device = torch.device("cpu")
    print("⚠️  No se detectó GPU, usando CPU (será más lento)")

# ============================================
# 2. CARGAR DATOS
# ============================================

print(f"\n📥 Cargando datos desde df_twitter.csv...")

try:
    df = pd.read_csv('df_twitter.csv', usecols=['Content'], nrows=1000)
except:
    df = pd.read_csv('df_twitter.csv', nrows=1000)
    if 'Content' not in df.columns:
        for col in df.columns:
            if 'content' in col.lower() or 'text' in col.lower():
                df = df.rename(columns={col: 'Content'})
                break

df['Content'] = df['Content'].astype(str).fillna('')
df_sample = df.sample(n=500, random_state=42).copy() if len(df) > 500 else df.copy()

tweets = df_sample['Content'].tolist()
print(f"✅ {len(tweets)} tweets cargados para análisis")

# ============================================
# 3. DEFINIR FRASES EQUIVALENTES POR CATEGORÍA
# ============================================

# DEFINICIÓN COMPLETA DE FRASES EQUIVALENTES
# Cada categoría tiene múltiples formas de expresar lo mismo
categorias_frases = {
    # Grupo 1: Lavado de manos
    'lavado_manos': [
        "Este texto habla del lavado de manos con agua y jabón",
        "Este texto menciona lavarse las manos por 20 segundos",
        "Este texto trata sobre la higiene de manos con jabón",
        "Este texto se refiere al lavado frecuente de manos",
        "Este texto discute la importancia de lavarse las manos"
    ],
    
    # Grupo 2: Alcohol gel / Desinfectante
    'alcohol_gel': [
        "Este texto habla del uso de alcohol en gel",
        "Este texto menciona el uso de desinfectante de manos",
        "Este texto trata sobre gel antibacterial",
        "Este texto se refiere al sanitizer para manos",
        "Este texto discute el uso de alcohol para desinfectar"
    ],
    
    # Grupo 3: Tapabocas / Mascarilla
    'tapabocas': [
        "Este texto habla del uso de tapabocas",
        "Este texto menciona el uso de mascarilla",
        "Este texto trata sobre cubrebocas o barbijo",
        "Este texto se refiere a la protección facial",
        "Este texto discute el uso de máscara protectora",
        "Este texto habla de usar protección respiratoria"
    ],
    
    # Grupo 4: Distanciamiento
    'distanciamiento': [
        "Este texto habla del distanciamiento social",
        "Este texto menciona mantener 2 metros de distancia",
        "Este texto trata sobre el alejamiento social",
        "Este texto se refiere a guardar distancia de seguridad",
        "Este texto discute mantener espacio personal"
    ],
    
    # Grupo 5: Teletrabajo
    'teletrabajo': [
        "Este texto habla de trabajar desde casa",
        "Este texto menciona el teletrabajo",
        "Este texto trata sobre home office",
        "Este texto se refiere al trabajo remoto",
        "Este texto discute trabajar en casa"
    ],
    
    # Grupo 6: Cuarentena
    'cuarentena': [
        "Este texto habla de cuarentena",
        "Este texto menciona aislamiento social",
        "Este texto trata sobre confinamiento",
        "Este texto se refiere a autoaislamiento",
        "Este texto discute encierro preventivo"
    ],
    
    # Grupo 7: Evitar reuniones
    'evitar_reuniones': [
        "Este texto habla de evitar reuniones sociales",
        "Este texto menciona no asistir a fiestas",
        "Este texto trata sobre evitar encuentros",
        "Este texto se refiere a no juntarse con gente",
        "Este texto discute evitar aglomeraciones"
    ],
    
    # Grupo 8: Evitar transporte público
    'evitar_transporte': [
        "Este texto habla de evitar transporte público",
        "Este texto menciona no usar metro o bus",
        "Este texto trata sobre evitar autobuses",
        "Este texto se refiere a no viajar en transporte masivo",
        "Este texto discute evitar trenes o subtes"
    ]
}

# Crear lista plana de TODAS las frases para el modelo
todas_las_frases = []
mapeo_frase_a_categoria = {}

for categoria, frases in categorias_frases.items():
    for frase in frases:
        todas_las_frases.append(frase)
        mapeo_frase_a_categoria[frase] = categoria

print(f"\n📚 Configuración de categorías:")
print(f"   • Categorías principales: {len(categorias_frases)}")
print(f"   • Frases equivalentes totales: {len(todas_las_frases)}")
print(f"   • Promedio por categoría: {len(todas_las_frases)//len(categorias_frases)} frases")

# ============================================
# 4. CARGAR MODELO ZERO-SHOT
# ============================================

print("\n🚀 Cargando modelo zero-shot...")

try:
    classifier = pipeline(
        "zero-shot-classification",
        model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
        device=0 if torch.cuda.is_available() else -1,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        batch_size=4 if torch.cuda.is_available() else 1
    )
    print("✅ Modelo cargado correctamente")
except Exception as e:
    print(f"⚠️  Error cargando modelo principal: {e}")
    print("   Cargando modelo más ligero...")
    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=0 if torch.cuda.is_available() else -1
    )

# ============================================
# 5. FUNCIÓN DE CLASIFICACIÓN CON FRASES EQUIVALENTES
# ============================================

def clasificar_con_frases_equivalentes(tweets, batch_size=4):
    """
    Clasifica tweets usando múltiples frases equivalentes por categoría.
    Devuelve todas las categorías detectadas con sus frases.
    """
    print(f"\n🎯 Clasificando {len(tweets)} tweets...")
    print(f"   Frases a evaluar por tweet: {len(todas_las_frases)}")
    
    resultados = []
    
    # Dividir en lotes para procesamiento eficiente
    batches = [tweets[i:i + batch_size] for i in range(0, len(tweets), batch_size)]
    
    for batch_idx, batch in enumerate(tqdm(batches, desc="Procesando lotes")):
        try:
            # Procesar lote completo
            batch_results = classifier(
                batch,
                candidate_labels=todas_las_frases,
                multi_label=True,
                hypothesis_template="{}",
                truncation=True,
                max_length=512
            )
            
            # Procesar cada tweet del lote
            for tweet, result in zip(batch, batch_results):
                categorias_detectadas = {}
                frases_detectadas = {}
                
                # Procesar cada frase clasificada
                for frase, score in zip(result['labels'], result['scores']):
                    # Usar umbral dinámico
                    if score > 0.3:  # Umbral base
                        categoria = mapeo_frase_a_categoria[frase]
                        
                        # Agregar a categorías detectadas
                        if categoria not in categorias_detectadas:
                            categorias_detectadas[categoria] = {
                                'mejor_score': score,
                                'frases': []
                            }
                        
                        # Agregar la frase específica
                        categorias_detectadas[categoria]['frases'].append({
                            'frase': frase,
                            'score': float(score)
                        })
                        
                        # Mantener el mejor score
                        if score > categorias_detectadas[categoria]['mejor_score']:
                            categorias_detectadas[categoria]['mejor_score'] = score
                
                # Filtrar categorías con score suficientemente alto
                categorias_finales = []
                detalles_frases = []
                
                for cat, info in categorias_detectadas.items():
                    if info['mejor_score'] > 0.35:  # Umbral final
                        categorias_finales.append(cat)
                        
                        # Ordenar frases por score
                        frases_ordenadas = sorted(info['frases'], 
                                                 key=lambda x: x['score'], 
                                                 reverse=True)
                        detalles_frases.append({
                            'categoria': cat,
                            'frases': frases_ordenadas[:3]  # Top 3 frases
                        })
                
                # Guardar resultado
                resultados.append({
                    'tweet': tweet,
                    'categorias': categorias_finales,
                    'num_categorias': len(categorias_finales),
                    'detalles_frases': detalles_frases,
                    'score_promedio': sum(info['mejor_score'] for info in categorias_detectadas.values()) / max(len(categorias_detectadas), 1)
                })
                
        except Exception as e:
            print(f"\n⚠️  Error en lote {batch_idx}: {str(e)[:100]}")
            # Procesar individualmente
            for tweet in batch:
                try:
                    result = classifier(
                        tweet,
                        candidate_labels=todas_las_frases[:50],  # Limitar para evitar errores
                        multi_label=True
                    )
                    
                    cats_detectadas = set()
                    for frase, score in zip(result['labels'], result['scores']):
                        if score > 0.4:
                            cat = mapeo_frase_a_categoria.get(frase)
                            if cat:
                                cats_detectadas.add(cat)
                    
                    resultados.append({
                        'tweet': tweet,
                        'categorias': list(cats_detectadas),
                        'num_categorias': len(cats_detectadas)
                    })
                    
                except:
                    resultados.append({
                        'tweet': tweet,
                        'categorias': [],
                        'num_categorias': 0
                    })
        
        # Limpiar memoria periódicamente
        if torch.cuda.is_available() and batch_idx % 10 == 0:
            torch.cuda.empty_cache()
            gc.collect()
    
    return resultados

# ============================================
# 6. EJECUTAR CLASIFICACIÓN
# ============================================

print("\n" + "="*60)
print("🤖 CLASIFICANDO CON FRASES EQUIVALENTES")
print("="*60)

import time
inicio = time.time()

resultados = clasificar_con_frases_equivalentes(tweets, batch_size=4)

fin = time.time()
print(f"\n⏱️  Tiempo total: {fin - inicio:.2f} segundos")
print(f"   Velocidad: {len(tweets)/(fin - inicio):.2f} tweets/segundo")

# ============================================
# 7. PROCESAR Y ANALIZAR RESULTADOS
# ============================================

print("\n" + "="*60)
print("📊 ANÁLISIS DE RESULTADOS")
print("="*60)

# Estadísticas básicas
tweets_con_medidas = sum(1 for r in resultados if r['num_categorias'] > 0)
porcentaje_con_medidas = (tweets_con_medidas / len(resultados)) * 100

print(f"\n📈 ESTADÍSTICAS GENERALES:")
print(f"   • Tweets analizados: {len(resultados)}")
print(f"   • Tweets con al menos 1 medida: {tweets_con_medidas} ({porcentaje_con_medidas:.1f}%)")
print(f"   • Tweets sin medidas detectadas: {len(resultados) - tweets_con_medidas}")

# Distribución por número de categorías
print(f"\n🔢 DISTRIBUCIÓN POR NÚMERO DE MEDIDAS:")
distribucion = {}
for r in resultados:
    num = r['num_categorias']
    if num >= 5:
        num = '5+'
    distribucion[num] = distribucion.get(num, 0) + 1

for num in sorted(distribucion.keys(), key=lambda x: (isinstance(x, str), x)):
    count = distribucion[num]
    porcentaje = (count / len(resultados)) * 100
    print(f"   • {num} medida(s): {count:3d} tweets ({porcentaje:5.1f}%)")

# Frecuencia de categorías
from collections import Counter

todas_categorias = []
for r in resultados:
    todas_categorias.extend(r['categorias'])

if todas_categorias:
    print(f"\n🏆 FRECUENCIA DE CATEGORÍAS:")
    for cat, count in Counter(todas_categorias).most_common():
        nombre = cat.replace('_', ' ').title()
        porcentaje = (count / len(resultados)) * 100
        print(f"   • {nombre:25s}: {count:3d} tweets ({porcentaje:5.1f}%)")

# ============================================
# 8. GUARDAR RESULTADOS DETALLADOS
# ============================================

print(f"\n💾 GUARDANDO RESULTADOS DETALLADOS...")

# DataFrame con resultados básicos
df_basico = pd.DataFrame([{
    'tweet': r['tweet'],
    'categorias': ', '.join(r['categorias']),
    'num_categorias': r['num_categorias'],
    'score_promedio': r.get('score_promedio', 0)
} for r in resultados])

df_basico.to_csv('resultados_categorias_basico.csv', index=False, encoding='utf-8')

# DataFrame con detalles de frases
datos_detallados = []
for r in resultados:
    if 'detalles_frases' in r:
        for detalle in r['detalles_frases']:
            for frase_info in detalle['frases']:
                datos_detallados.append({
                    'tweet': r['tweet'][:200],  # Acortar para el CSV
                    'categoria': detalle['categoria'],
                    'frase_detectada': frase_info['frase'],
                    'score_frase': frase_info['score']
                })

if datos_detallados:
    df_detallado = pd.DataFrame(datos_detallados)
    df_detallado.to_csv('resultados_frases_detallado.csv', index=False, encoding='utf-8')

# Resumen estadístico
resumen = {
    'total_tweets': len(resultados),
    'tweets_con_medidas': tweets_con_medidas,
    'porcentaje_con_medidas': porcentaje_con_medidas,
    'tiempo_total_seg': fin - inicio,
    'categorias_usadas': len(categorias_frases),
    'frases_equivalentes': len(todas_las_frases)
}

# Agregar frecuencia por categoría
for cat, count in Counter(todas_categorias).most_common():
    resumen[f'freq_{cat}'] = count
    resumen[f'perc_{cat}'] = (count / len(resultados)) * 100

df_resumen = pd.DataFrame([resumen])
df_resumen.to_csv('resumen_estadistico.csv', index=False)

print(f"✅ Archivos guardados:")
print(f"   • resultados_categorias_basico.csv - Clasificación básica")
print(f"   • resultados_frases_detallado.csv - Frases específicas detectadas")
print(f"   • resumen_estadistico.csv - Estadísticas completas")

# ============================================
# 9. MOSTRAR EJEMPLOS DETALLADOS
# ============================================

print("\n" + "="*60)
print("📝 EJEMPLOS DETALLADOS DE CLASIFICACIÓN")
print("="*60)

# Encontrar tweets interesantes
tweets_multicategoria = [r for r in resultados if r['num_categorias'] >= 2]
tweets_con_detalles = [r for r in resultados if 'detalles_frases' in r and r['detalles_frases']]

if tweets_multicategoria:
    print(f"\n🌟 TWEETS CON MÚLTIPLES CATEGORÍAS (Top 3):")
    for i, r in enumerate(tweets_multicategoria[:3], 1):
        print(f"\n   🏅 Ejemplo {i} - {r['num_categorias']} categorías:")
        print(f"      Tweet: \"{r['tweet'][:120]}...\"")
        print(f"      Categorías: {', '.join(r['categorias'])}")
        
        if 'detalles_frases' in r:
            print(f"      Frases detectadas:")
            for detalle in r['detalles_frases'][:2]:  # Mostrar primeras 2 categorías
                mejor_frase = detalle['frases'][0]['frase'] if detalle['frases'] else 'N/A'
                print(f"        • {detalle['categoria']}: \"{mejor_frase[:60]}...\"")

# Mostrar cómo se detectaron frases equivalentes
if tweets_con_detalles:
    print(f"\n🔍 EJEMPLO DE FRASES EQUIVALENTES DETECTADAS:")
    tweet_ejemplo = tweets_con_detalles[0]
    
    print(f"\n   Tweet: \"{tweet_ejemplo['tweet'][:100]}...\"")
    
    for detalle in tweet_ejemplo['detalles_frases']:
        print(f"\n   📌 Categoría: {detalle['categoria'].replace('_', ' ').title()}")
        for frase_info in detalle['frases'][:2]:  # Mostrar 2 mejores frases
            print(f"      • \"{frase_info['frase']}\"")
            print(f"        Score: {frase_info['score']:.3f}")

# ============================================
# 10. ANÁLISIS DE FRASES MÁS EFECTIVAS
# ============================================

print("\n" + "="*60)
print("📈 ANÁLISIS DE FRASES MÁS EFECTIVAS POR CATEGORÍA")
print("="*60)

if datos_detallados:
    df_frases = pd.DataFrame(datos_detallados)
    
    print(f"\n🎯 FRASES CON MEJORES SCORES POR CATEGORÍA:")
    
    for categoria in categorias_frases.keys():
        frases_cat = df_frases[df_frases['categoria'] == categoria]
        
        if not frases_cat.empty:
            # Agrupar por frase y calcular estadísticas
            stats_frases = frases_cat.groupby('frase_detectada').agg({
                'score_frase': ['mean', 'count', 'max']
            }).round(3)
            
            stats_frases.columns = ['score_promedio', 'frecuencia', 'score_max']
            stats_frases = stats_frases.sort_values('score_promedio', ascending=False)
            
            top_frase = stats_frases.iloc[0]
            nombre_cat = categoria.replace('_', ' ').title()
            
            print(f"\n   {nombre_cat}:")
            print(f"      Frase más efectiva: \"{stats_frases.index[0][:70]}...\"")
            print(f"      Score promedio: {top_frase['score_promedio']:.3f}")
            print(f"      Veces detectada: {top_frase['frecuencia']}")

# ============================================
# 11. LIMPIAR MEMORIA
# ============================================

print(f"\n🧹 LIMPIANDO MEMORIA...")
del classifier
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE")
print(f"   Se detectaron {len(set(todas_categorias))} categorías diferentes")
print(f"   Usando {len(todas_las_frases)} frases equivalentes para mejor cobertura")

🔍 Verificando hardware disponible...
PyTorch version: 2.9.1+cu126
CUDA disponible: True
✅ GPU detectada: NVIDIA GeForce RTX 4050 Laptop GPU

📥 Cargando datos desde df_twitter.csv...
✅ 500 tweets cargados para análisis

📚 Configuración de categorías:
   • Categorías principales: 8
   • Frases equivalentes totales: 41
   • Promedio por categoría: 5 frases

🚀 Cargando modelo zero-shot...


config.json: 0.00B [00:00, ?B/s]

c:\Users\afpue\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\afpue\.cache\huggingface\hub\models--MoritzLaurer--deberta-v3-large-zeroshot-v2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

Device set to use cuda:0


✅ Modelo cargado correctamente

🤖 CLASIFICANDO CON FRASES EQUIVALENTES

🎯 Clasificando 500 tweets...
   Frases a evaluar por tweet: 41


Procesando lotes: 100%|██████████| 125/125 [05:58<00:00,  2.87s/it]



⏱️  Tiempo total: 358.62 segundos
   Velocidad: 1.39 tweets/segundo

📊 ANÁLISIS DE RESULTADOS

📈 ESTADÍSTICAS GENERALES:
   • Tweets analizados: 500
   • Tweets con al menos 1 medida: 195 (39.0%)
   • Tweets sin medidas detectadas: 305

🔢 DISTRIBUCIÓN POR NÚMERO DE MEDIDAS:
   • 0 medida(s): 305 tweets ( 61.0%)
   • 1 medida(s):  96 tweets ( 19.2%)
   • 2 medida(s):  52 tweets ( 10.4%)
   • 3 medida(s):  41 tweets (  8.2%)
   • 4 medida(s):   6 tweets (  1.2%)

🏆 FRECUENCIA DE CATEGORÍAS:
   • Cuarentena               : 170 tweets ( 34.0%)
   • Distanciamiento          :  94 tweets ( 18.8%)
   • Evitar Reuniones         :  39 tweets (  7.8%)
   • Tapabocas                :  23 tweets (  4.6%)
   • Teletrabajo              :  18 tweets (  3.6%)
   • Alcohol Gel              :   2 tweets (  0.4%)
   • Lavado Manos             :   1 tweets (  0.2%)

💾 GUARDANDO RESULTADOS DETALLADOS...
✅ Archivos guardados:
   • resultados_categorias_basico.csv - Clasificación básica
   • resultados_fras